In [4]:
pip install statsmodels --user


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
import matplotlib.patches as mpatches
from matplotlib.ticker import PercentFormatter

# =========================
# Paths (adjust if needed)
# =========================
super_path = "/Users/judycheng/Desktop/supercharger in washington state.xls"
residents_path = "/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx"
ev_path = "/Users/judycheng/Desktop/coordinates_output.xlsm"

# =========================
# Step 1: Read Excel files
# =========================
super_df = pd.read_excel(super_path)
residents_df = pd.read_excel(residents_path)
ev_df = pd.read_excel(ev_path)

# =================================
# Step 2: Filter to King County data
# =================================
king_super = super_df[(super_df["State"] == "Washington") & (super_df["County"] == "King County")]
king_residents = residents_df[residents_df["County"] == "King County"]
king_ev = ev_df[(ev_df["State"] == "WA") & (ev_df["County"] == "King")]

population_column = "total"
population_25_59 = float(king_residents[population_column].values[0])
num_chargers = int(len(king_super))
num_evs = int(len(king_ev))

print(f"King County population (25-59): {population_25_59:,.0f}")
print(f"Existing Superchargers: {num_chargers}")
print(f"Existing EVs (rows in EV file for King): {num_evs}")

# ==========================
# Step 3: Policy targets
# ==========================
policy_targets = {2030: 0.45, 2035: 0.60, 2040: 0.70, 2050: 0.95}

# =================================================
# Step 4: Charger target math
# =================================================
king_county_area = 2307  # sq mi
radius_miles = 2
area_per_charger = np.pi * radius_miles**2
lower_chargers_goal = int(np.ceil(king_county_area / area_per_charger))  # 2-mile geo-coverage
upper_chargers_goal = int(round(population_25_59 / 1500.0))              # 1 per 1,500 residents

print(f"Lower bound chargers (geo coverage): {lower_chargers_goal}")
print(f"Upper bound chargers (1 per 1,500 residents): {upper_chargers_goal}")

# ====================
# Step 5: Build scenarios
# ====================
years = list(range(2025, 2051))
n_years = len(years)
rows = []

# ---- Lower: slow Phase 2, catch-up Phase 3 ----
current_lower = num_chargers
total_lower_needed = max(0, lower_chargers_goal - num_chargers)
yearly_build_lower = int(round(total_lower_needed / n_years)) if n_years > 0 else 0

for y in years:
    # Chargers build (constant toward geo cap)
    new_lower = yearly_build_lower
    if current_lower + new_lower > lower_chargers_goal:
        new_lower = lower_chargers_goal - current_lower
    current_lower += max(0, new_lower)

    # Adoption: P1 modest; P2 slower than Upper; P3 catch-up
    if y <= 2030:
        eff = 0.10 + (policy_targets[2030] - 0.10) * ((y - 2025) / 5) * 0.35
    elif y <= 2040:
        eff_start = 0.10 + (policy_targets[2030] - 0.10) * 0.35
        eff = eff_start + (policy_targets[2040] * 0.70 - eff_start) * ((y - 2030) / 10)
    else:
        eff_start = policy_targets[2040] * 0.70
        eff = eff_start + (policy_targets[2050] - eff_start) * ((y - 2040) / 10)

    rows.append({
        "Scenario": "Lower",
        "Year": y,
        "Total_Chargers": int(current_lower),
        "New_Chargers": int(max(0, new_lower)),
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999))
    })

# ---- Upper: 3-phase slowdown, aggressive to meet targets ----
current_upper = num_chargers
total_upper_needed = max(0, upper_chargers_goal - num_chargers)

# Allocate adds by phase (front-load then taper)
phase_weights = []
for y in years:
    if 2025 <= y <= 2030: phase_weights.append(1.0)
    elif 2031 <= y <= 2040: phase_weights.append(0.6)
    else: phase_weights.append(0.2)
phase_weights = np.array(phase_weights)
phase_weights = phase_weights / phase_weights.sum()
yearly_adds = np.round(phase_weights * total_upper_needed).astype(int)

# Ensure exact sum
diff = total_upper_needed - yearly_adds.sum()
if diff != 0:
    idx = 0
    step = 1 if diff > 0 else -1
    for _ in range(abs(diff)):
        yearly_adds[idx] += step
        idx = (idx + 1) % len(yearly_adds)

for y, add in zip(years, yearly_adds):
    current_upper += max(0, int(add))

    if y <= 2030:
        eff = 0.10 + (policy_targets[2030] - 0.10) * ((y - 2025) / 5)
    elif y <= 2035:
        eff = policy_targets[2030] + (policy_targets[2035] - policy_targets[2030]) * ((y - 2030) / 5)
    elif y <= 2040:
        eff = policy_targets[2035] + (policy_targets[2040] - policy_targets[2035]) * ((y - 2035) / 5)
    else:
        eff = policy_targets[2040] + (policy_targets[2050] - policy_targets[2040]) * ((y - 2040) / 10)

    rows.append({
        "Scenario": "Upper",
        "Year": y,
        "Total_Chargers": int(current_upper),
        "New_Chargers": int(max(0, int(add))),
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999))
    })

sc_df = pd.DataFrame(rows)

# ===============================
# Step 6: Monte Carlo (Monotone; 2030–2035 lean to Lower; post-2035 accelerate <95%)
# ===============================
yrs = sc_df["Year"].unique()
lo = sc_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Lower"].values
hi = sc_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Upper"].values

# --- 6A) Target path the MC reverts toward (NOT midpoint)
# 2025–2030: slightly neutral (0.55 toward Upper)
# 2030–2035: lean closer to Lower (0.25 toward Upper)
# 2036–2050: accelerate toward Upper (0.70) but cap below 95%
target_w_upper = []
for y in yrs:
    if 2025 <= y <= 2030:   target_w_upper.append(0.55)
    elif 2031 <= y <= 2035: target_w_upper.append(0.25)
    else:                   target_w_upper.append(0.70)
target_w_upper = np.array(target_w_upper, dtype=float)

raw_target = (1 - target_w_upper) * lo + target_w_upper * hi
target_cap = 0.94  # keep MC under 95% by 2050; tweak if desired
target = np.minimum(raw_target, target_cap)
target = np.maximum.accumulate(target)  # ensure the target itself is non-decreasing

# --- 6B) Public charger growth & home-charger dampening (for context, modest role here)
w_upper_charger = []
for y in yrs:
    if 2025 <= y <= 2030:   w_upper_charger.append(0.80)
    elif 2031 <= y <= 2040: w_upper_charger.append(0.30)
    else:                   w_upper_charger.append(0.55)
w_upper_charger = np.array(w_upper_charger, dtype=float)

lowC = sc_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Lower"].values
hiC  = sc_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Upper"].values
C = (1 - w_upper_charger) * lowC + w_upper_charger * hiC
C_growth = np.r_[0, np.diff(C)]
den = np.maximum(1.0, np.r_[C[0], C[:-1]])
C_growth_rate = np.divide(C_growth, den, out=np.zeros_like(C_growth, dtype=float), where=den>0)
C_growth_rate = np.clip(C_growth_rate, 0, 0.50)

# Home-charger share
home_share = []
for y in yrs:
    if 2025 <= y <= 2030:   home_share.append(0.55 + (0.72 - 0.55) * (y - 2025) / 5)
    elif 2031 <= y <= 2040: home_share.append(0.72 + (0.90 - 0.72) * (y - 2030) / 10)
    else:                   home_share.append(0.90 + (0.93 - 0.90) * (y - 2040) / 10)
home_share = np.array(home_share, dtype=float)

# --- 6C) Monotonic MC simulation in rate space (never decreases)
N = 1000
T = len(yrs)
rng = np.random.default_rng(42)

# Phase parameters: lean-low & flat in 2030–2035; accelerate afterward
kappa = []  # pull toward target
beta  = []  # charger push (dampened by home_share)
sigma = []  # small noise
for y in yrs:
    if 2025 <= y <= 2030:
        kappa.append(0.30); beta.append(0.45); sigma.append(0.015)
    elif 2031 <= y <= 2035:
        kappa.append(0.25); beta.append(0.20); sigma.append(0.010)  # lean-low, flatter
    elif 2036 <= y <= 2040:
        kappa.append(0.45); beta.append(0.30); sigma.append(0.018)  # begin acceleration
    else:
        kappa.append(0.55); beta.append(0.30); sigma.append(0.020)  # sustained acceleration
kappa = np.array(kappa, dtype=float)
beta  = np.array(beta, dtype=float) * (1 - home_share)
sigma = np.array(sigma, dtype=float)

paths = np.zeros((N, T), dtype=float)
start_level = float(np.clip(0.35 * lo[0] + 0.65 * hi[0], 0.0, target_cap))  # start upper-leaning but capped
paths[:, 0] = start_level

for t in range(1, T):
    prev = paths[:, t-1]
    mean_revert = kappa[t] * (target[t] - prev)     # >0 if below target
    charger_push = beta[t] * C_growth_rate[t]       # >= 0
    noise = rng.normal(0.0, sigma[t], size=N)       # small, zero-mean
    delta = mean_revert + charger_push + noise
    delta = np.maximum(delta, 0.0)                  # forbid negative increments
    next_rate = prev + delta
    next_rate = np.maximum(next_rate, lo[t])        # never below Lower for that year
    next_rate = np.minimum(next_rate, target_cap)   # keep <95%
    next_rate = np.maximum(next_rate, prev)         # enforce monotonicity
    paths[:, t] = next_rate

# Percentiles per year
p10 = np.percentile(paths, 10, axis=0)
p50 = np.percentile(paths, 50, axis=0)
p90 = np.percentile(paths, 90, axis=0)

ev_p10 = (p10 * population_25_59).astype(int)
ev_p50 = (p50 * population_25_59).astype(int)
ev_p90 = (p90 * population_25_59).astype(int)

# ===========================================
# Step 7: Build final table with new columns
# ===========================================
wide = sc_df.pivot(index="Year", columns="Scenario", values=["Total_Chargers", "Adoption_Rate"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

wide["Forecast_Adoption_P10"] = p10
wide["Forecast_Adoption_P50"] = p50
wide["Forecast_Adoption_P90"] = p90
wide["Forecast_EVs_P10"] = ev_p10
wide["Forecast_EVs_P50"] = ev_p50
wide["Forecast_EVs_P90"] = ev_p90

# Budget-weighted charger path to show as "Forecast"
budget_weighted_C = ((1 - w_upper_charger) * lowC + w_upper_charger * hiC).round().astype(int)
wide["Forecast_Chargers"] = budget_weighted_C

# Optional check: how often P50 sits within [Lower, Upper]
within_band = ((wide["Forecast_Adoption_P50"] >= wide["Adoption_Rate_Lower"]) &
               (wide["Forecast_Adoption_P50"] <= wide["Adoption_Rate_Upper"]))
wide["Forecast_within_bounds"] = within_band.astype(int)

# =================================
# Step 8: Save results to Excel
# =================================
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "king_county_ev_projection_mc_monotonic.xlsx")
with pd.ExcelWriter(desktop_path, engine="openpyxl") as writer:
    sc_df.to_excel(writer, sheet_name="Scenarios_LU", index=False)
    wide.to_excel(writer, sheet_name="Forecast", index=False)

# ==============================
# Step 9: Charts (with phases)
# ==============================
phase_spans = [(2025, 2030, "Phase 1"), (2030, 2040, "Phase 2"), (2040, 2050, "Phase 3")]
phase_colors = ["green", "yellow", "orange"]

# --- A) Total Chargers: Lower vs Upper vs Budget-weighted Forecast ---
plt.figure(figsize=(10, 6))
ax1 = plt.gca()
ax1.plot(wide["Year"], wide["Total_Chargers_Lower"], marker="o", label="Lower Bound")
ax1.plot(wide["Year"], wide["Total_Chargers_Upper"], marker="o", label="Upper Bound")
ax1.plot(wide["Year"], wide["Forecast_Chargers"], linestyle="--", linewidth=2, label="Budget-Weighted Forecast")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax1.axvspan(start, end, color=c, alpha=0.10)
line_handles, line_labels = ax1.get_legend_handles_labels()
phase_handles = [mpatches.Patch(color=c, alpha=0.10, label=name) for c, (_, _, name) in zip(phase_colors, phase_spans)]
ax1.legend(line_handles + phase_handles, line_labels + [name for *_, name in phase_spans], loc="best")
ax1.set_xlabel("Year")
ax1.set_ylabel("Total Chargers")
ax1.set_title("King County: EV Chargers — Lower vs Upper vs Budget-Weighted")
ax1.grid(True)
plt.tight_layout()
charger_chart = os.path.join(os.path.expanduser("~"), "Desktop", "charger_projection_mc_monotonic.png")
plt.savefig(charger_chart)
plt.close()

# --- B) EV Adoption Rate: Lower vs Upper vs Monotonic MC ---
plt.figure(figsize=(10, 6))
ax2 = plt.gca()
ax2.plot(wide["Year"], wide["Adoption_Rate_Lower"], marker="o", label="Lower Bound")
ax2.plot(wide["Year"], wide["Adoption_Rate_Upper"], marker="o", label="Upper Bound")
ax2.fill_between(wide["Year"], wide["Forecast_Adoption_P10"], wide["Forecast_Adoption_P90"],
                 alpha=0.20, label="MC Forecast Band (P10–P90)")
ax2.plot(wide["Year"], wide["Forecast_Adoption_P50"], linestyle="--", linewidth=2, label="MC Median (P50)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax2.axvspan(start, end, color=c, alpha=0.10)
line_handles, line_labels = ax2.get_legend_handles_labels()
phase_handles = [mpatches.Patch(color=c, alpha=0.10, label=name) for c, (_, _, name) in zip(phase_colors, phase_spans)]
ax2.legend(line_handles + phase_handles, line_labels + [name for *_, name in phase_spans], loc="best")
ax2.set_xlabel("Year")
ax2.set_ylabel("EV Adoption Rate")
ax2.set_title("King County: EV Adoption — Lower vs Upper vs Monotonic MC")
ax2.grid(True)
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
adoption_chart = os.path.join(os.path.expanduser("~"), "Desktop", "ev_adoption_rate_mc_monotonic.png")
plt.savefig(adoption_chart)
plt.close()

# --- C) EV Registrations: MC P10/P50/P90 ---
plt.figure(figsize=(10, 6))
ax3 = plt.gca()
ax3.fill_between(wide["Year"], wide["Forecast_EVs_P10"], wide["Forecast_EVs_P90"],
                 alpha=0.20, label="MC EVs Band (P10–P90)")
ax3.plot(wide["Year"], wide["Forecast_EVs_P50"], linestyle="--", linewidth=2, label="MC EVs Median (P50)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax3.axvspan(start, end, color=c, alpha=0.10)
ax3.legend(loc="best")
ax3.set_xlabel("Year")
ax3.set_ylabel("EV Registrations (count)")
ax3.set_title("King County: EV Registrations — Monotonic MC P10 / P50 / P90")
ax3.grid(True)
plt.tight_layout()
evs_chart = os.path.join(os.path.expanduser("~"), "Desktop", "ev_registrations_mc_monotonic.png")
plt.savefig(evs_chart)
plt.close()

# ===================================
# Step 10: Embed charts back to Excel
# ===================================
wb = load_workbook(desktop_path)
ws = wb.create_sheet(title="Charts")
ws.add_image(Image(charger_chart), "A1")
ws.add_image(Image(adoption_chart), "A40")
ws.add_image(Image(evs_chart), "A79")
wb.save(desktop_path)

# Summary printout
pct_within = 100.0 * wide["Forecast_within_bounds"].mean()
print(f"\n✅ Forecast complete. Excel and charts saved to: {desktop_path}")
print(f"• Median adoption within bounds in {pct_within:.1f}% of years.")
for yr in [2030, 2035, 2040, 2050]:
    r = wide.loc[wide['Year'] == yr].iloc[0]
    print(f"  - {yr}: P50={r['Forecast_Adoption_P50']:.2%}, "
          f"Lower={r['Adoption_Rate_Lower']:.2%}, Upper={r['Adoption_Rate_Upper']:.2%}, "
          f"EVs_P50={int(r['Forecast_EVs_P50']):,}")


King County population (25-59): 1,241,805
Existing Superchargers: 13
Existing EVs (rows in EV file for King): 101838
Lower bound chargers (geo coverage): 184
Upper bound chargers (1 per 1,500 residents): 828

✅ Forecast complete. Excel and charts saved to: /Users/judycheng/Desktop/king_county_ev_projection_mc_monotonic.xlsx
• Median adoption within bounds in 84.6% of years.
  - 2030: P50=37.09%, Lower=22.25%, Upper=45.00%, EVs_P50=460,587
  - 2035: P50=40.03%, Lower=35.62%, Upper=60.00%, EVs_P50=497,092
  - 2040: P50=61.00%, Lower=49.00%, Upper=70.00%, EVs_P50=757,554
  - 2050: P50=94.00%, Lower=95.00%, Upper=95.00%, EVs_P50=1,167,296


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import math
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from matplotlib.ticker import PercentFormatter
import matplotlib.patches as mpatches

# --- Step 1: Read Excel files ---
super_df = pd.read_excel("/Users/judycheng/Desktop/supercharger in washington state.xls")
residents_df = pd.read_excel("/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx")
ev_df = pd.read_excel("/Users/judycheng/Desktop/coordinates_output.xlsm")

# --- Step 2: Pierce County data ---
pierce_super = super_df[(super_df["State"] == "Washington") & (super_df["County"] == "Pierce County")]
pierce_residents = residents_df[residents_df["County"] == "Pierce County"]
pierce_ev = ev_df[(ev_df["State"] == "WA") & (ev_df["County"] == "Pierce")]

population_column = "total"
population_25_59 = float(pierce_residents[population_column].values[0])
num_chargers = int(len(pierce_super))
num_evs = int(len(pierce_ev))

print(f"Pierce County population (25-59): {population_25_59:,.0f}")
print(f"Existing Superchargers (from Excel): {num_chargers}")
print(f"Existing EVs: {num_evs}")

# --- Step 3: Policy targets ---
policy_targets = {2030: 0.45, 2035: 0.60, 2040: 0.70, 2050: 0.95}

# --- Step 4: Lower and Upper Bound charger targets ---
pierce_county_area = 1790  # sq mi
radius_miles = 3
area_per_charger = np.pi * radius_miles**2  # ~28.27 sq mi per charger
lower_chargers = int(np.ceil(pierce_county_area / area_per_charger))  # geo coverage (3-mile radius)

upper_base_target = population_25_59 / 2500.0  # 1 per 2500 residents

print(f"Lower bound chargers (geo coverage, 3-mile radius): {lower_chargers}")
print(f"Original upper bound (1/2500 residents): {upper_base_target:.0f}")

# --- Step 5: Build scenarios (Lower below Upper in P2; Lower reaches 95% by 2050) ---
years = list(range(2025, 2051))
results = []

# LOWER BOUND (constant build per year, rounded up)
current_lower = num_chargers
total_lower_needed = max(0, lower_chargers - num_chargers)
n_years = len(years)
yearly_build_lower_exact = total_lower_needed / n_years  # keep as float

print("\n--- LOWER BOUND DEBUG ---")
print(f"Existing chargers (from Excel): {num_chargers}")
print(f"Target chargers (3-mile coverage): {lower_chargers}")
print(f"Total additional chargers needed: {total_lower_needed}")
print(f"Exact average build per year: {yearly_build_lower_exact:.2f}")
print("---------------------------\n")

for i, y in enumerate(years):
    # Round up build to nearest whole number; adjust last year to hit target exactly
    new_lower = math.ceil(yearly_build_lower_exact)
    if current_lower + new_lower > lower_chargers or i == len(years) - 1:
        new_lower = lower_chargers - current_lower
    current_lower += new_lower

    # Lower adoption shape: modest P1, conservative P2 (below Upper), catch-up to 95% by 2050
    if y <= 2030:
        eff = 0.10 + (policy_targets[2030] - 0.10) * ((y - 2025) / 5) * 0.35
    elif y <= 2040:
        eff_start = 0.10 + (policy_targets[2030] - 0.10) * 0.35
        eff = eff_start + (policy_targets[2040] * 0.70 - eff_start) * ((y - 2030) / 10)
    else:
        eff_start = policy_targets[2040] * 0.70
        eff = eff_start + (policy_targets[2050] - eff_start) * ((y - 2040) / 10)

    results.append({
        "Scenario": "Lower",
        "Year": y,
        "Total_Chargers": int(current_lower),
        "New_Chargers": int(new_lower),
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999))
    })

print(f"✅ Lower-bound chargers total at 2050: {int(current_lower)} (target: {lower_chargers})\n")

# UPPER BOUND (3-phase slowdown)
base_build_rate = upper_base_target / n_years
current_upper = num_chargers

for y in years:
    if 2025 <= y <= 2030:
        yearly_add = int(round(base_build_rate * 1.0))
    elif 2031 <= y <= 2040:
        yearly_add = int(round(base_build_rate * 0.6))
    elif 2041 <= y <= 2050:
        yearly_add = int(round(base_build_rate * 0.2))
    else:
        yearly_add = 0

    current_upper += yearly_add

    # Upper adoption (tracks policy milestones)
    if y <= 2030:
        eff = 0.10 + (policy_targets[2030] - 0.10) * ((y - 2025) / 5)
    elif y <= 2035:
        eff = policy_targets[2030] + (policy_targets[2035] - policy_targets[2030]) * ((y - 2030) / 5)
    elif y <= 2040:
        eff = policy_targets[2035] + (policy_targets[2040] - policy_targets[2035]) * ((y - 2035) / 5)
    else:
        eff = policy_targets[2040] + (policy_targets[2050] - policy_targets[2040]) * ((y - 2040) / 10)

    results.append({
        "Scenario": "Upper",
        "Year": y,
        "Total_Chargers": int(current_upper),
        "New_Chargers": yearly_add,
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999))
    })

forecast_df = pd.DataFrame(results)

# --- Step 6: Monte Carlo (Monotone; P2 leans to Lower 2030–2035; post-2035 accelerates but <95%) ---

def clamp(x, lo=0.0, hi=0.9999):
    return float(np.clip(x, lo, hi))

yrs = np.array(years)
lo_path = forecast_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Lower"].values
hi_path = forecast_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Upper"].values

# 6A) Set the *target* trajectory the MC reverts toward (NOT midpoint):
# - Phase 1 (2025–2030): neutral-to-slightly low (w=0.55 toward Upper)
# - Phase 2a (2030–2035): closer to Lower (w=0.25 upper weight)
# - Phase 2b/Phase 3 (2036–2050): accelerate toward Upper but cap below 95% (e.g., 0.94)
target_w_upper = []
for y in yrs:
    if 2025 <= y <= 2030:
        target_w_upper.append(0.55)
    elif 2031 <= y <= 2035:
        target_w_upper.append(0.25)  # closer to Lower in early Phase 2
    else:
        target_w_upper.append(0.70)  # stronger pull up after 2035
target_w_upper = np.array(target_w_upper)

raw_target = (1 - target_w_upper) * lo_path + target_w_upper * hi_path
# cap the ultimate target at <95% to satisfy "still lower than target 95%"
target_cap = 0.94
target = np.minimum(raw_target, target_cap)
# enforce target to be non-decreasing year-over-year
target = np.maximum.accumulate(target)

# 6B) Public charger growth (budget-weighted path) + home share to modulate impact
w_upper_charger = []
for y in yrs:
    if 2025 <= y <= 2030: w_upper_charger.append(0.80)  # strong DCFC in Phase 1
    elif 2031 <= y <= 2040: w_upper_charger.append(0.30)  # slower public build (home focus)
    else: w_upper_charger.append(0.55)  # modest ramp late
w_upper_charger = np.array(w_upper_charger)

lowC = forecast_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Lower"].values
hiC  = forecast_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Upper"].values
C = (1 - w_upper_charger) * lowC + w_upper_charger * hiC
C_growth = np.r_[0, np.diff(C)]
den = np.maximum(1.0, np.r_[C[0], C[:-1]])
C_growth_rate = np.divide(C_growth, den, out=np.zeros_like(C_growth, dtype=float), where=den>0)
C_growth_rate = np.clip(C_growth_rate, 0, 0.50)

# home charging share dampens the effect of public chargers
home_share = []
for y in yrs:
    if 2025 <= y <= 2030:   home_share.append(0.55 + (0.72 - 0.55) * (y - 2025) / 5)  # 55%→72%
    elif 2031 <= y <= 2040: home_share.append(0.72 + (0.90 - 0.72) * (y - 2030) / 10) # 72%→90%
    else:                   home_share.append(0.90 + (0.93 - 0.90) * (y - 2040) / 10) # 90%→93%
home_share = np.array(home_share)

# 6C) Monotonic MC simulation in *rate space* (not logit), so we can force non-decreasing:
# delta_t = max(0, mean_reversion + charger_push + noise)
# rate_t = min(rate_{t-1} + delta_t, target_cap), and also >= lower bound
N = 1000
T = len(yrs)
rng = np.random.default_rng(42)

# Phase-varying parameters:
# - Stronger pull in 2036+ to accelerate
# - Charger impact damped by (1 - home_share)
# - Smaller noise in 2030–2035 (flatter lean-low)
kappa = []  # pull toward target (per year)
beta  = []  # sensitivity to public DCFC growth
sigma = []  # random noise
for y in yrs:
    if 2025 <= y <= 2030:
        kappa.append(0.30); beta.append(0.45); sigma.append(0.015)
    elif 2031 <= y <= 2035:
        kappa.append(0.25); beta.append(0.20); sigma.append(0.010)  # lean-low, flatter
    elif 2036 <= y <= 2040:
        kappa.append(0.45); beta.append(0.30); sigma.append(0.018)  # begin acceleration
    else:
        kappa.append(0.55); beta.append(0.30); sigma.append(0.020)  # sustained acceleration
kappa = np.array(kappa, dtype=float)
beta  = np.array(beta, dtype=float) * (1 - home_share)
sigma = np.array(sigma, dtype=float)

paths = np.zeros((N, T), dtype=float)
# Start at a small offset above 2025 lower bound, but below upper
start_level = clamp(0.35 * lo_path[0] + 0.65 * hi_path[0], 0.0, target_cap)
paths[:, 0] = start_level

for t in range(1, T):
    prev = paths[:, t-1]
    mean_revert = kappa[t] * (target[t] - prev)          # positive if below target
    charger_push = beta[t] * C_growth_rate[t]            # >= 0
    noise = rng.normal(0.0, sigma[t], size=N)            # can be +/- but we'll clip total
    delta = mean_revert + charger_push + noise
    delta = np.maximum(delta, 0.0)                       # enforce nonnegative increment
    next_rate = prev + delta
    # keep path >= lower bound that year, and <= target cap
    next_rate = np.maximum(next_rate, lo_path[t])
    next_rate = np.minimum(next_rate, target_cap)
    # enforce monotonicity explicitly
    next_rate = np.maximum(next_rate, prev)
    paths[:, t] = next_rate

# Percentiles per year
p10 = np.percentile(paths, 10, axis=0)
p50 = np.percentile(paths, 50, axis=0)   # <= 0.94 by construction
p90 = np.percentile(paths, 90, axis=0)

ev_p10 = (p10 * population_25_59).astype(int)
ev_p50 = (p50 * population_25_59).astype(int)
ev_p90 = (p90 * population_25_59).astype(int)

# --- Step 7: Save results (with forecast columns) to Excel ---
wide = forecast_df.pivot(index="Year", columns="Scenario", values=["Total_Chargers", "Adoption_Rate"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

wide["Forecast_Adoption_P10"] = p10
wide["Forecast_Adoption_P50"] = p50
wide["Forecast_Adoption_P90"] = p90
wide["Forecast_EVs_P10"] = ev_p10
wide["Forecast_EVs_P50"] = ev_p50
wide["Forecast_EVs_P90"] = ev_p90

# Budget-weighted charger path to display as a "forecast" for chargers (optional)
budget_weighted_C = ((1 - w_upper_charger) * lowC + w_upper_charger * hiC).round().astype(int)
wide["Forecast_Chargers"] = budget_weighted_C

desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "pierce_county_ev_projection_mc_monotonic.xlsx")
with pd.ExcelWriter(desktop_path, engine="openpyxl") as writer:
    forecast_df.to_excel(writer, sheet_name="Scenarios_LU", index=False)
    wide.to_excel(writer, sheet_name="Forecast", index=False)

# --- Step 8: Charts ---
phase_spans = [(2025, 2030, "Phase 1"), (2030, 2040, "Phase 2"), (2040, 2050, "Phase 3")]
phase_colors = ["green", "yellow", "orange"]

# A) Total Chargers
plt.figure(figsize=(10,6))
ax1 = plt.gca()
ax1.plot(wide["Year"], wide["Total_Chargers_Lower"], marker="o", label="Lower Bound")
ax1.plot(wide["Year"], wide["Total_Chargers_Upper"], marker="o", label="Upper Bound")
ax1.plot(wide["Year"], wide["Forecast_Chargers"], linestyle="--", linewidth=2, label="Budget-Weighted Forecast")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax1.axvspan(start, end, color=c, alpha=0.10)
line_handles, line_labels = ax1.get_legend_handles_labels()
phase_handles = [mpatches.Patch(color=c, alpha=0.10, label=name) for c, (_, _, name) in zip(phase_colors, phase_spans)]
ax1.legend(line_handles + phase_handles, line_labels + [name for *_, name in phase_spans], loc="best")
ax1.set_xlabel("Year")
ax1.set_ylabel("Total Chargers")
ax1.set_title("Pierce County: EV Charger Projection (Lower vs Upper vs Budget-Weighted)")
ax1.grid(True)
plt.tight_layout()
charger_chart = os.path.join(os.path.expanduser("~"), "Desktop", "pierce_charger_projection_mc_monotonic.png")
plt.savefig(charger_chart)
plt.close()

# B) EV Adoption Rate (with phases + MC P10/P50/P90)
plt.figure(figsize=(10,6))
ax2 = plt.gca()
# Bounds
ax2.plot(wide["Year"], wide["Adoption_Rate_Lower"], marker="o", label="Lower Bound")
ax2.plot(wide["Year"], wide["Adoption_Rate_Upper"], marker="o", label="Upper Bound")
# MC band + median (monotonic)
ax2.fill_between(wide["Year"], wide["Forecast_Adoption_P10"], wide["Forecast_Adoption_P90"],
                 alpha=0.20, label="MC Forecast Band (P10–P90)")
ax2.plot(wide["Year"], wide["Forecast_Adoption_P50"], linestyle="--", linewidth=2, label="MC Median (P50)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax2.axvspan(start, end, color=c, alpha=0.10)
line_handles, line_labels = ax2.get_legend_handles_labels()
phase_handles = [mpatches.Patch(color=c, alpha=0.10, label=name) for c, (_, _, name) in zip(phase_colors, phase_spans)]
ax2.legend(line_handles + phase_handles, line_labels + [name for *_, name in phase_spans], loc="best")
ax2.set_xlabel("Year")
ax2.set_ylabel("EV Adoption Rate")
ax2.set_title("Pierce County: EV Adoption — Lower vs Upper vs Monotonic MC")
ax2.grid(True)
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
adoption_chart = os.path.join(os.path.expanduser("~"), "Desktop", "pierce_ev_adoption_rate_mc_monotonic.png")
plt.savefig(adoption_chart)
plt.close()

# C) EV Registrations (P10/P50/P90)
plt.figure(figsize=(10,6))
ax3 = plt.gca()
ax3.fill_between(wide["Year"], wide["Forecast_EVs_P10"], wide["Forecast_EVs_P90"],
                 alpha=0.20, label="MC EVs Band (P10–P90)")
ax3.plot(wide["Year"], wide["Forecast_EVs_P50"], linestyle="--", linewidth=2, label="MC EVs Median (P50)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax3.axvspan(start, end, color=c, alpha=0.10)
ax3.legend(loc="best")
ax3.set_xlabel("Year")
ax3.set_ylabel("EV Registrations (count)")
ax3.set_title("Pierce County: EV Registrations — Monotonic MC P10 / P50 / P90")
ax3.grid(True)
plt.tight_layout()
evs_chart = os.path.join(os.path.expanduser("~"), "Desktop", "pierce_ev_registrations_mc_monotonic.png")
plt.savefig(evs_chart)
plt.close()

# --- Step 9: Embed charts in Excel ---
wb = load_workbook(desktop_path)
ws = wb.create_sheet(title="Charts")
ws.add_image(Image(charger_chart), "A1")
ws.add_image(Image(adoption_chart), "A40")
ws.add_image(Image(evs_chart), "A79")
wb.save(desktop_path)

print(f"\n✅ Projection complete. Excel and charts saved to: {desktop_path}")
print(f"Final 2050 upper bound chargers (running total): {int(current_upper)}")
print(f"P50 adoption in 2050: {p50[-1]:.2%} (kept below 95% target)")


Pierce County population (25-59): 448,201
Existing Superchargers (from Excel): 1
Existing EVs: 16277
Lower bound chargers (geo coverage, 3-mile radius): 64
Original upper bound (1/2500 residents): 179

--- LOWER BOUND DEBUG ---
Existing chargers (from Excel): 1
Target chargers (3-mile coverage): 64
Total additional chargers needed: 63
Exact average build per year: 2.42
---------------------------

✅ Lower-bound chargers total at 2050: 64 (target: 64)


✅ Projection complete. Excel and charts saved to: /Users/judycheng/Desktop/pierce_county_ev_projection_mc_monotonic.xlsx
Final 2050 upper bound chargers (running total): 93
P50 adoption in 2050: 94.00% (kept below 95% target)


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import math
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from matplotlib.ticker import PercentFormatter
import matplotlib.patches as mpatches

# --- Step 1: Read Excel files ---
super_df = pd.read_excel("/Users/judycheng/Desktop/supercharger in washington state.xls")
residents_df = pd.read_excel("/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx")
ev_df = pd.read_excel("/Users/judycheng/Desktop/coordinates_output.xlsm")

# --- Step 2: Kitsap County data ---
kitsap_super = super_df[(super_df["State"] == "Washington") & (super_df["County"] == "Kitsap County")]
kitsap_residents = residents_df[residents_df["County"] == "Kitsap County"]
kitsap_ev = ev_df[(ev_df["State"] == "WA") & (ev_df["County"] == "Kitsap")]

population_column = "total"
population_25_59 = float(kitsap_residents[population_column].values[0])
num_chargers = int(len(kitsap_super))
num_evs = int(len(kitsap_ev))

print(f"Kitsap County population (25-59): {population_25_59:,.0f}")
print(f"Existing Superchargers (from Excel): {num_chargers}")
print(f"Existing EVs: {num_evs}")

# --- Step 3: Policy targets ---
policy_targets = {2030: 0.45, 2035: 0.60, 2040: 0.70, 2050: 0.95}

# --- Step 4: Lower and Upper Bound charger targets ---
kitsap_county_area = 566  # sq mi
radius_miles = 5
area_per_charger = np.pi * radius_miles**2  # ~78.54 sq mi per charger
lower_chargers = int(np.ceil(kitsap_county_area / area_per_charger))  # geo coverage (5-mile radius)

upper_base_target = population_25_59 / 2500.0  # 1 per 2,500 residents

print(f"Lower bound chargers (geo coverage, 5-mile radius): {lower_chargers}")
print(f"Original upper bound (1/2500 residents): {upper_base_target:.0f}")

# --- Step 5: Build scenarios (Lower below Upper in P2; Lower reaches 95% by 2050) ---
years = list(range(2025, 2051))
n_years = len(years)
results = []

# LOWER BOUND (constant build per year, rounded up)
current_lower = num_chargers
total_lower_needed = max(0, lower_chargers - num_chargers)
yearly_build_lower_exact = total_lower_needed / n_years  # float, not int

print("\n--- LOWER BOUND DEBUG ---")
print(f"Existing chargers (from Excel): {num_chargers}")
print(f"Target chargers (5-mile coverage): {lower_chargers}")
print(f"Total additional chargers needed: {total_lower_needed}")
print(f"Exact average build per year: {yearly_build_lower_exact:.2f}")
print("---------------------------\n")

for i, y in enumerate(years):
    # Round up build per year; adjust last year to hit target exactly
    new_lower = math.ceil(yearly_build_lower_exact)
    if current_lower + new_lower > lower_chargers or i == len(years) - 1:
        new_lower = lower_chargers - current_lower
    current_lower += new_lower

    # Lower adoption shape:
    #   P1: modest,
    #   P2: deliberately below Upper (lean-conservative),
    #   P3: accelerate to 95% by 2050
    if y <= 2030:
        eff = 0.10 + (policy_targets[2030] - 0.10) * ((y - 2025) / 5) * 0.35
    elif y <= 2040:
        eff_start = 0.10 + (policy_targets[2030] - 0.10) * 0.35
        eff = eff_start + (policy_targets[2040] * 0.70 - eff_start) * ((y - 2030) / 10)
    else:
        eff_start = policy_targets[2040] * 0.70
        eff = eff_start + (policy_targets[2050] - eff_start) * ((y - 2040) / 10)

    results.append({
        "Scenario": "Lower",
        "Year": y,
        "Total_Chargers": int(current_lower),
        "New_Chargers": int(new_lower),
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999))
    })

print(f"✅ Lower-bound chargers total at 2050: {int(current_lower)} (target: {lower_chargers})\n")

# UPPER BOUND (3-phase slowdown)
base_build_rate = upper_base_target / n_years
current_upper = num_chargers

for y in years:
    if 2025 <= y <= 2030:
        yearly_add = int(round(base_build_rate * 1.0))
    elif 2031 <= y <= 2040:
        yearly_add = int(round(base_build_rate * 0.6))
    elif 2041 <= y <= 2050:
        yearly_add = int(round(base_build_rate * 0.2))
    else:
        yearly_add = 0

    current_upper += yearly_add

    # Upper adoption (tracks policy milestones more directly)
    if y <= 2030:
        eff = 0.10 + (policy_targets[2030] - 0.10) * ((y - 2025) / 5)
    elif y <= 2035:
        eff = policy_targets[2030] + (policy_targets[2035] - policy_targets[2030]) * ((y - 2030) / 5)
    elif y <= 2040:
        eff = policy_targets[2035] + (policy_targets[2040] - policy_targets[2035]) * ((y - 2035) / 5)
    else:
        eff = policy_targets[2040] + (policy_targets[2050] - policy_targets[2040]) * ((y - 2040) / 10)

    results.append({
        "Scenario": "Upper",
        "Year": y,
        "Total_Chargers": int(current_upper),
        "New_Chargers": yearly_add,
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999))
    })

forecast_df = pd.DataFrame(results)

# --- Step 6: Monte Carlo (Monotonic; 2030–2035 near Lower; post-2035 accelerates but <95%) ---

def clamp(x, lo=0.0, hi=0.9999):
    return float(np.clip(x, lo, hi))

yrs = np.array(years)
lo_path = forecast_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Lower"].values
hi_path = forecast_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Upper"].values

# 6A) MC target path (NOT midpoint):
#   2025–2030: neutral/slightly high (0.55 toward Upper)
#   2030–2035: closer to Lower (0.25 toward Upper)
#   2036–2050: accelerate (0.70 toward Upper) but cap <95%
target_w_upper = []
for y in yrs:
    if 2025 <= y <= 2030:
        target_w_upper.append(0.55)
    elif 2031 <= y <= 2035:
        target_w_upper.append(0.25)
    else:
        target_w_upper.append(0.70)
target_w_upper = np.array(target_w_upper)

raw_target = (1 - target_w_upper) * lo_path + target_w_upper * hi_path
target_cap = 0.94  # keep MC end below 95%; tweak as needed
target = np.minimum(raw_target, target_cap)
target = np.maximum.accumulate(target)  # ensure non-decreasing target

# 6B) Budget-weighted charger context + home-charger dampening (for plausible dynamics)
w_upper_charger = []
for y in yrs:
    if 2025 <= y <= 2030:   w_upper_charger.append(0.80)  # Phase 1: strong DCFC build
    elif 2031 <= y <= 2040: w_upper_charger.append(0.30)  # Phase 2: slower public build, home focus
    else:                   w_upper_charger.append(0.55)  # Phase 3: modest catch-up
w_upper_charger = np.array(w_upper_charger)

lowC = forecast_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Lower"].values
hiC  = forecast_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Upper"].values
C = (1 - w_upper_charger) * lowC + w_upper_charger * hiC
C_growth = np.r_[0, np.diff(C)]
den = np.maximum(1.0, np.r_[C[0], C[:-1]])
C_growth_rate = np.divide(C_growth, den, out=np.zeros_like(C_growth, dtype=float), where=den>0)
C_growth_rate = np.clip(C_growth_rate, 0, 0.50)

# Rising home charging share reduces marginal impact of public DCFC
home_share = []
for y in yrs:
    if 2025 <= y <= 2030:   home_share.append(0.55 + (0.72 - 0.55) * (y - 2025) / 5)
    elif 2031 <= y <= 2040: home_share.append(0.72 + (0.90 - 0.72) * (y - 2030) / 10)
    else:                   home_share.append(0.90 + (0.93 - 0.90) * (y - 2040) / 10)
home_share = np.array(home_share)

# 6C) Monotonic MC simulation directly in rate space
N = 1000
T = len(yrs)
rng = np.random.default_rng(42)

# Phase parameters: flatter 2030–2035; stronger acceleration after 2035
kappa = []  # pull toward target
beta  = []  # charger push (dampened by home_share)
sigma = []  # small noise
for y in yrs:
    if 2025 <= y <= 2030:
        kappa.append(0.30); beta.append(0.45); sigma.append(0.015)
    elif 2031 <= y <= 2035:
        kappa.append(0.25); beta.append(0.20); sigma.append(0.010)
    elif 2036 <= y <= 2040:
        kappa.append(0.45); beta.append(0.30); sigma.append(0.018)
    else:
        kappa.append(0.55); beta.append(0.30); sigma.append(0.020)
kappa = np.array(kappa, dtype=float)
beta  = np.array(beta, dtype=float) * (1 - home_share)
sigma = np.array(sigma, dtype=float)

paths = np.zeros((N, T), dtype=float)
start_level = clamp(0.35 * lo_path[0] + 0.65 * hi_path[0], 0.0, target_cap)  # start leaning upper but capped
paths[:, 0] = start_level

for t in range(1, T):
    prev = paths[:, t-1]
    mean_revert = kappa[t] * (target[t] - prev)     # >0 if below target
    charger_push = beta[t] * C_growth_rate[t]       # >= 0
    noise = rng.normal(0.0, sigma[t], size=N)       # small random
    delta = mean_revert + charger_push + noise
    delta = np.maximum(delta, 0.0)                  # forbid negative increments
    next_rate = prev + delta
    next_rate = np.maximum(next_rate, lo_path[t])   # never below Lower for that year
    next_rate = np.minimum(next_rate, target_cap)   # keep <95%
    next_rate = np.maximum(next_rate, prev)         # enforce monotonicity
    paths[:, t] = next_rate

# Percentiles per year
p10 = np.percentile(paths, 10, axis=0)
p50 = np.percentile(paths, 50, axis=0)
p90 = np.percentile(paths, 90, axis=0)

ev_p10 = (p10 * population_25_59).astype(int)
ev_p50 = (p50 * population_25_59).astype(int)
ev_p90 = (p90 * population_25_59).astype(int)

# --- Step 7: Save results (with forecast columns) to Excel ---
wide = forecast_df.pivot(index="Year", columns="Scenario", values=["Total_Chargers", "Adoption_Rate"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

wide["Forecast_Adoption_P10"] = p10
wide["Forecast_Adoption_P50"] = p50
wide["Forecast_Adoption_P90"] = p90
wide["Forecast_EVs_P10"] = ev_p10
wide["Forecast_EVs_P50"] = ev_p50
wide["Forecast_EVs_P90"] = ev_p90

# Budget-weighted charger path as "forecast"
budget_weighted_C = ((2 - 2) * lowC + (2 - 2) * hiC)  # placeholder no-op to keep code consistent
budget_weighted_C = ((1 - w_upper_charger) * lowC + w_upper_charger * hiC).round().astype(int)
wide["Forecast_Chargers"] = budget_weighted_C

desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "kitsap_county_ev_projection_mc_monotonic.xlsx")
with pd.ExcelWriter(desktop_path, engine="openpyxl") as writer:
    forecast_df.to_excel(writer, sheet_name="Scenarios_LU", index=False)
    wide.to_excel(writer, sheet_name="Forecast", index=False)

# --- Step 8: Charts (with phase shading) ---
phase_spans = [(2025, 2030, "Phase 1"), (2030, 2040, "Phase 2"), (2040, 2050, "Phase 3")]
phase_colors = ["green", "yellow", "orange"]

# A) Total Chargers
plt.figure(figsize=(10,6))
ax1 = plt.gca()
ax1.plot(wide["Year"], wide["Total_Chargers_Lower"], marker="o", label="Lower Bound")
ax1.plot(wide["Year"], wide["Total_Chargers_Upper"], marker="o", label="Upper Bound")
ax1.plot(wide["Year"], wide["Forecast_Chargers"], linestyle="--", linewidth=2, label="Budget-Weighted Forecast")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax1.axvspan(start, end, color=c, alpha=0.10)
line_handles, line_labels = ax1.get_legend_handles_labels()
phase_handles = [mpatches.Patch(color=c, alpha=0.10, label=name) for c, (_, _, name) in zip(phase_colors, phase_spans)]
ax1.legend(line_handles + phase_handles, line_labels + [name for *_, name in phase_spans], loc="best")
ax1.set_xlabel("Year")
ax1.set_ylabel("Total Chargers")
ax1.set_title("Kitsap County: EV Chargers — Lower vs Upper vs Budget-Weighted")
ax1.grid(True)
plt.tight_layout()
charger_chart = os.path.join(os.path.expanduser("~"), "Desktop", "kitsap_charger_projection_mc_monotonic.png")
plt.savefig(charger_chart)
plt.close()

# B) EV Adoption Rate (with phases + Monotonic MC)
plt.figure(figsize=(10,6))
ax2 = plt.gca()
# Bounds
ax2.plot(wide["Year"], wide["Adoption_Rate_Lower"], marker="o", label="Lower Bound")
ax2.plot(wide["Year"], wide["Adoption_Rate_Upper"], marker="o", label="Upper Bound")
# MC band + median
ax2.fill_between(wide["Year"], wide["Forecast_Adoption_P10"], wide["Forecast_Adoption_P90"],
                 alpha=0.20, label="MC Forecast Band (P10–P90)")
ax2.plot(wide["Year"], wide["Forecast_Adoption_P50"], linestyle="--", linewidth=2, label="MC Median (P50)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax2.axvspan(start, end, color=c, alpha=0.10)
line_handles, line_labels = ax2.get_legend_handles_labels()
phase_handles = [mpatches.Patch(color=c, alpha=0.10, label=name) for c, (_, _, name) in zip(phase_colors, phase_spans)]
ax2.legend(line_handles + phase_handles, line_labels + [name for *_, name in phase_spans], loc="best")
ax2.set_xlabel("Year")
ax2.set_ylabel("EV Adoption Rate")
ax2.set_title("Kitsap County: EV Adoption — Lower vs Upper vs Monotonic MC")
ax2.grid(True)
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
adoption_chart = os.path.join(os.path.expanduser("~"), "Desktop", "kitsap_ev_adoption_rate_mc_monotonic.png")
plt.savefig(adoption_chart)
plt.close()

# C) EV Registrations (P10/P50/P90)
plt.figure(figsize=(10,6))
ax3 = plt.gca()
ax3.fill_between(wide["Year"], wide["Forecast_EVs_P10"], wide["Forecast_EVs_P90"],
                 alpha=0.20, label="MC EVs Band (P10–P90)")
ax3.plot(wide["Year"], wide["Forecast_EVs_P50"], linestyle="--", linewidth=2, label="MC EVs Median (P50)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax3.axvspan(start, end, color=c, alpha=0.10)
ax3.legend(loc="best")
ax3.set_xlabel("Year")
ax3.set_ylabel("EV Registrations (count)")
ax3.set_title("Kitsap County: EV Registrations — Monotonic MC P10 / P50 / P90")
ax3.grid(True)
plt.tight_layout()
evs_chart = os.path.join(os.path.expanduser("~"), "Desktop", "kitsap_ev_registrations_mc_monotonic.png")
plt.savefig(evs_chart)
plt.close()

# --- Step 9: Embed charts in Excel ---
wb = load_workbook(desktop_path)
ws = wb.create_sheet(title="Charts")
ws.add_image(Image(charger_chart), "A1")
ws.add_image(Image(adoption_chart), "A40")
ws.add_image(Image(evs_chart), "A79")
wb.save(desktop_path)

print(f"\n✅ Projection complete. Excel and charts saved to: {desktop_path}")
print(f"Final 2050 upper bound chargers (running total): {int(current_upper)}")
print(f"P50 adoption in 2050 (should be <95%): {p50[-1]:.2%}")


Kitsap County population (25-59): 125,820
Existing Superchargers (from Excel): 0
Existing EVs: 6428
Lower bound chargers (geo coverage, 5-mile radius): 8
Original upper bound (1/2500 residents): 50

--- LOWER BOUND DEBUG ---
Existing chargers (from Excel): 0
Target chargers (5-mile coverage): 8
Total additional chargers needed: 8
Exact average build per year: 0.31
---------------------------

✅ Lower-bound chargers total at 2050: 8 (target: 8)


✅ Projection complete. Excel and charts saved to: /Users/judycheng/Desktop/kitsap_county_ev_projection_mc_monotonic.xlsx
Final 2050 upper bound chargers (running total): 22
P50 adoption in 2050 (should be <95%): 94.00%


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
import math
import matplotlib.patches as mpatches
from matplotlib.ticker import PercentFormatter

# --- Step 1: Read Excel files ---
super_df = pd.read_excel("/Users/judycheng/Desktop/supercharger in washington state.xls")
residents_df = pd.read_excel("/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx")
ev_df = pd.read_excel("/Users/judycheng/Desktop/coordinates_output.xlsm")

# --- Step 2: Chelan County data ---
chelan_super = super_df[(super_df["State"] == "Washington") & (super_df["County"] == "Chelan County")]
chelan_residents = residents_df[residents_df["County"] == "Chelan County"]
chelan_ev = ev_df[(ev_df["State"] == "WA") & (ev_df["County"] == "Chelan")]

population_column = "total"
population_25_59 = float(chelan_residents[population_column].values[0])
num_chargers = int(len(chelan_super))  # existing chargers
num_evs = int(len(chelan_ev))

print(f"Chelan County population (25-59): {population_25_59:,.0f}")
print(f"Existing Superchargers: {num_chargers}")
print(f"Existing EVs: {num_evs}")

# --- Step 3: Policy targets ---
policy_targets = {2030: 0.45, 2035: 0.60, 2040: 0.70, 2050: 0.95}

# --- Step 4: Lower and Upper Bound charger targets ---
county_area = 2965  # sq mi
radius_miles = 15
lower_chargers = math.ceil(county_area / (np.pi * radius_miles**2))  # ~15-mi radius coverage
upper_chargers = math.ceil(population_25_59 / 5000)  # 1 per 5,000 residents (label fixed)

print(f"Lower bound chargers (geo coverage, 15-mi radius): {lower_chargers}")
print(f"Upper bound chargers (1 per 5,000 residents): {upper_chargers}")

# --- Step 5: Build Lower/Upper scenarios (Lower < Upper in Phase 2; Lower hits 95% by 2050) ---
years = list(range(2025, 2051))
results = []

# LOWER: front-loaded to reach geo target, adoption below Upper in 2030–2040, catch-up to 95% by 2050
current_lower = num_chargers
total_lower_needed = max(0, lower_chargers - num_chargers)
n_years = len(years)
yearly_build_lower = int(round(total_lower_needed / n_years)) if n_years > 0 else 0
yearly_build_lower = max(1, yearly_build_lower) if total_lower_needed > 0 else 0  # at least 1 until hit

for i, y in enumerate(years):
    # charger build
    new_lower = yearly_build_lower if current_lower < lower_chargers else 0
    if current_lower + new_lower > lower_chargers:
        new_lower = lower_chargers - current_lower
    current_lower += max(0, new_lower)

    # adoption shape
    if y <= 2030:
        eff = 0.10 + (policy_targets[2030] - 0.10) * ((y - 2025) / 5) * 0.35
    elif y <= 2040:
        eff_start = 0.10 + (policy_targets[2030] - 0.10) * 0.35
        eff = eff_start + (policy_targets[2040] * 0.70 - eff_start) * ((y - 2030) / 10)  # keep < Upper
    else:
        eff_start = policy_targets[2040] * 0.70
        eff = eff_start + (policy_targets[2050] - eff_start) * ((y - 2040) / 10)          # reaches 95%

    results.append({
        "Scenario": "Lower",
        "Year": y,
        "Total_Chargers": int(current_lower),
        "New_Chargers": int(max(0, new_lower)),
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999)),
        "EVs": int(population_25_59 * np.clip(eff, 0.0, 0.9999))
    })

# UPPER: smooth path to population-based charger target; adoption tracks policy milestones more directly
current_upper = num_chargers
total_upper_needed = max(0, upper_chargers - num_chargers)
# simple back-loaded average to 2050
for y in years:
    remaining = max(0, upper_chargers - current_upper)
    remaining_years = max(1, 2050 - y + 1)
    yearly_add = int(math.ceil(remaining / remaining_years)) if remaining > 0 else 0
    current_upper += yearly_add

    if y <= 2030:
        eff = 0.10 + (policy_targets[2030] - 0.10) * ((y - 2025) / 5)
    elif y <= 2035:
        eff = policy_targets[2030] + (policy_targets[2035] - policy_targets[2030]) * ((y - 2030) / 5)
    elif y <= 2040:
        eff = policy_targets[2035] + (policy_targets[2040] - policy_targets[2035]) * ((y - 2035) / 5)
    else:
        eff = policy_targets[2040] + (policy_targets[2050] - policy_targets[2040]) * ((y - 2040) / 10)

    results.append({
        "Scenario": "Upper",
        "Year": y,
        "Total_Chargers": int(current_upper),
        "New_Chargers": int(yearly_add),
        "Adoption_Rate": float(np.clip(eff, 0.0, 0.9999)),
        "EVs": int(population_25_59 * np.clip(eff, 0.0, 0.9999))
    })

sc_df = pd.DataFrame(results)

# --- Step 6: Monte Carlo forecast (Monotonic; 2030–2035 near Lower; accelerate post-2035, end <95%) ---

def clamp(x, lo=0.0, hi=0.9999):
    return float(np.clip(x, lo, hi))

yrs = sc_df["Year"].unique()
lo_path = sc_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Lower"].values
hi_path = sc_df.pivot(index="Year", columns="Scenario", values="Adoption_Rate")["Upper"].values

# 6A) Target path (NOT midpoint) with weights over phases
# 2025–2030: slightly upper-leaning (0.55)
# 2031–2035: closer to lower (0.25)
# 2036–2050: accelerate (0.70) but cap <95%
target_w_upper = []
for y in yrs:
    if 2025 <= y <= 2030:   target_w_upper.append(0.55)
    elif 2031 <= y <= 2035: target_w_upper.append(0.25)
    else:                   target_w_upper.append(0.70)
target_w_upper = np.array(target_w_upper, dtype=float)

raw_target = (1 - target_w_upper) * lo_path + target_w_upper * hi_path
target_cap = 0.94  # keep MC under 95% by 2050
target = np.minimum(raw_target, target_cap)
target = np.maximum.accumulate(target)  # ensure target itself is non-decreasing

# 6B) Budget-weighted charger growth & home-charger share (context)
w_upper_charger = []
for y in yrs:
    if 2025 <= y <= 2030:   w_upper_charger.append(0.80)  # Phase 1: strong DCFC build
    elif 2031 <= y <= 2040: w_upper_charger.append(0.30)  # Phase 2: home-charger era
    else:                   w_upper_charger.append(0.55)  # Phase 3: modest catch-up
w_upper_charger = np.array(w_upper_charger, dtype=float)

lowC = sc_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Lower"].values
hiC  = sc_df.pivot(index="Year", columns="Scenario", values="Total_Chargers")["Upper"].values
C = (1 - w_upper_charger) * lowC + w_upper_charger * hiC
C_growth = np.r_[0, np.diff(C)]
den = np.maximum(1.0, np.r_[C[0], C[:-1]])
C_growth_rate = np.divide(C_growth, den, out=np.zeros_like(C_growth, dtype=float), where=den>0)
C_growth_rate = np.clip(C_growth_rate, 0, 0.50)

home_share = []
for y in yrs:
    if 2025 <= y <= 2030:   home_share.append(0.50 + (0.70 - 0.50) * (y - 2025) / 5)  # 50%→70%
    elif 2031 <= y <= 2040: home_share.append(0.70 + (0.88 - 0.70) * (y - 2030) / 10) # 70%→88%
    else:                   home_share.append(0.88 + (0.92 - 0.88) * (y - 2040) / 10) # 88%→92%
home_share = np.array(home_share, dtype=float)

# 6C) Monotonic MC simulation directly in rate space
N = 1000
T = len(yrs)
rng = np.random.default_rng(42)

# Phase parameters: flatter 2030–2035; stronger acceleration after 2035
kappa = []  # pull toward target
beta  = []  # charger push (dampened by home_share)
sigma = []  # small noise
for y in yrs:
    if 2025 <= y <= 2030:
        kappa.append(0.30); beta.append(0.45); sigma.append(0.015)
    elif 2031 <= y <= 2035:
        kappa.append(0.25); beta.append(0.20); sigma.append(0.010)
    elif 2036 <= y <= 2040:
        kappa.append(0.45); beta.append(0.30); sigma.append(0.018)
    else:
        kappa.append(0.55); beta.append(0.30); sigma.append(0.020)
kappa = np.array(kappa, dtype=float)
beta  = np.array(beta, dtype=float) * (1 - home_share)
sigma = np.array(sigma, dtype=float)

paths = np.zeros((N, T), dtype=float)
start_level = clamp(0.35 * lo_path[0] + 0.65 * hi_path[0], 0.0, target_cap)
paths[:, 0] = start_level

for t in range(1, T):
    prev = paths[:, t-1]
    mean_revert = kappa[t] * (target[t] - prev)   # positive if below target
    charger_push = beta[t] * C_growth_rate[t]     # >= 0
    noise = rng.normal(0.0, sigma[t], size=N)     # small +/- noise
    delta = np.maximum(mean_revert + charger_push + noise, 0.0)  # no negative increments
    next_rate = prev + delta
    next_rate = np.maximum(next_rate, lo_path[t])      # never below Lower for that year
    next_rate = np.minimum(next_rate, target_cap)      # keep <95%
    next_rate = np.maximum(next_rate, prev)            # enforce monotonicity
    paths[:, t] = next_rate

# Percentiles per year (adoption) and EV counts
p10 = np.percentile(paths, 10, axis=0)
p50 = np.percentile(paths, 50, axis=0)
p90 = np.percentile(paths, 90, axis=0)
ev_p10 = (p10 * population_25_59).astype(int)
ev_p50 = (p50 * population_25_59).astype(int)
ev_p90 = (p90 * population_25_59).astype(int)

# --- Step 7: Save results with forecast columns ---
wide = sc_df.pivot(index="Year", columns="Scenario", values=["Total_Chargers", "Adoption_Rate"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

# add MC forecasts
wide["Forecast_Adoption_P10"] = p10
wide["Forecast_Adoption_P50"] = p50
wide["Forecast_Adoption_P90"] = p90
wide["Forecast_EVs_P10"] = ev_p10
wide["Forecast_EVs_P50"] = ev_p50
wide["Forecast_EVs_P90"] = ev_p90

# Display a budget-weighted charger "forecast" for context
budget_weighted_C = ((1 - w_upper_charger) * lowC + w_upper_charger * hiC).round().astype(int)
wide["Forecast_Chargers"] = budget_weighted_C

desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "chelan_county_ev_projection_mc_monotonic.xlsx")
with pd.ExcelWriter(desktop_path, engine="openpyxl") as writer:
    sc_df.to_excel(writer, sheet_name="Scenarios_LU", index=False)
    wide.to_excel(writer, sheet_name="Forecast", index=False)

# --- Step 8: Charts (with phase shading) ---
phase_spans = [(2025, 2030, "Phase 1"), (2030, 2040, "Phase 2"), (2040, 2050, "Phase 3")]
phase_colors = ["green", "yellow", "orange"]

# 1) Total Chargers — Lower vs Upper vs Forecast
plt.figure(figsize=(10,6))
ax1 = plt.gca()
ax1.plot(wide["Year"], wide["Total_Chargers_Lower"], marker="o", label="Lower Bound")
ax1.plot(wide["Year"], wide["Total_Chargers_Upper"], marker="o", label="Upper Bound")
ax1.plot(wide["Year"], wide["Forecast_Chargers"], linestyle="--", linewidth=2, label="Budget-Weighted Forecast")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax1.axvspan(start, end, color=c, alpha=0.10)
line_handles, line_labels = ax1.get_legend_handles_labels()
phase_handles = [mpatches.Patch(color=c, alpha=0.10, label=name) for c, (_, _, name) in zip(phase_colors, phase_spans)]
ax1.legend(line_handles + phase_handles, line_labels + [name for *_, name in phase_spans], loc="best")
ax1.set_xlabel("Year"); ax1.set_ylabel("Total Chargers")
ax1.set_title("Chelan County: EV Chargers — Lower vs Upper vs Budget-Weighted")
ax1.grid(True); plt.tight_layout()
charger_chart = os.path.join(os.path.expanduser("~"), "Desktop", "chelan_chargers_mc_monotonic.png")
plt.savefig(charger_chart); plt.close()

# 2) EV Adoption Rate — Lower vs Upper vs Monotonic MC (with phase colors)
plt.figure(figsize=(10,6))
ax2 = plt.gca()
ax2.plot(wide["Year"], wide["Adoption_Rate_Lower"], marker="o", label="Lower Bound")
ax2.plot(wide["Year"], wide["Adoption_Rate_Upper"], marker="o", label="Upper Bound")
ax2.fill_between(wide["Year"], wide["Forecast_Adoption_P10"], wide["Forecast_Adoption_P90"],
                 alpha=0.20, label="MC Forecast Band (P10–P90)")
ax2.plot(wide["Year"], wide["Forecast_Adoption_P50"], linestyle="--", linewidth=2, label="MC Median (P50)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax2.axvspan(start, end, color=c, alpha=0.10)
line_handles, line_labels = ax2.get_legend_handles_labels()
phase_handles = [mpatches.Patch(color=c, alpha=0.10, label=name) for c, (_, _, name) in zip(phase_colors, phase_spans)]
ax2.legend(line_handles + phase_handles, line_labels + [name for *_, name in phase_spans], loc="best")
ax2.set_xlabel("Year"); ax2.set_ylabel("EV Adoption Rate")
ax2.set_title("Chelan County: EV Adoption — Lower vs Upper vs Monotonic MC")
ax2.grid(True); ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
adoption_chart = os.path.join(os.path.expanduser("~"), "Desktop", "chelan_adoption_mc_monotonic.png")
plt.savefig(adoption_chart); plt.close()

# 3) EV Registrations — Monotonic MC P10/P50/P90 (with phases)
plt.figure(figsize=(10,6))
ax3 = plt.gca()
ax3.fill_between(wide["Year"], wide["Forecast_EVs_P10"], wide["Forecast_EVs_P90"],
                 alpha=0.20, label="MC EVs Band (P10–P90)")
ax3.plot(wide["Year"], wide["Forecast_EVs_P50"], linestyle="--", linewidth=2, label="MC EVs Median (P50)")
for (start, end, name), c in zip(phase_spans, phase_colors):
    ax3.axvspan(start, end, color=c, alpha=0.10)
ax3.legend(loc="best"); ax3.set_xlabel("Year"); ax3.set_ylabel("EV Registrations (count)")
ax3.set_title("Chelan County: EV Registrations — Monotonic MC P10 / P50 / P90")
ax3.grid(True); plt.tight_layout()
evs_chart = os.path.join(os.path.expanduser("~"), "Desktop", "chelan_evs_mc_monotonic.png")
plt.savefig(evs_chart); plt.close()

# --- Step 9: Embed charts in Excel ---
wb = load_workbook(desktop_path)
ws = wb.create_sheet(title="Charts")
ws.add_image(Image(charger_chart), "A1")
ws.add_image(Image(adoption_chart), "A40")
ws.add_image(Image(evs_chart), "A79")
wb.save(desktop_path)

print(f"\n✅ Chelan County EV Projection (monotonic MC) saved to: {desktop_path}")
print(f"P50 adoption in 2050 (should be <95%): {p50[-1]:.2%}")


Chelan County population (25-59): 33,879
Existing Superchargers: 3
Existing EVs: 1250
Lower bound chargers (geo coverage, 15-mi radius): 5
Upper bound chargers (1 per 5,000 residents): 7

✅ Chelan County EV Projection (monotonic MC) saved to: /Users/judycheng/Desktop/chelan_county_ev_projection_mc_monotonic.xlsx
P50 adoption in 2050 (should be <95%): 94.00%
